# 17 — Growth-Shrinkage Audit (v2.4)

**LeukemiPrediag v2.4 — outcome-free.** Decides whether hierarchical / empirical-Bayes growth shrinkage is justified over the v2.3 notebook-16 baselines, using patient-held-out cross-fitting, 20 repeated CV seeds, leave-one-cohort-out transport, patient-cluster bootstrap CIs and split-conformal 80% intervals.

This validates **VAF forecasting only**. It does not establish prediction of AML/MDS or any clinical outcome. See `V24_MODEL_AUDIT.md`.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    CANDIDATES = [MYDRIVE / "leukemia-prediagnose" / "blood-cancer-early-detection",
                  MYDRIVE / "blood-cancer-early-detection"]
    CANDIDATES += list(MYDRIVE.glob("*/blood-cancer-early-detection"))
except ImportError:
    CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "src" / "growth_shrinkage.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"src/growth_shrinkage.py not found in any of: {CANDIDATES}")
os.chdir(PROJECT_ROOT)
SERIAL = PROJECT_ROOT / "data" / "processed" / "v2" / "real_serial_filtered.csv"
OUTDIR = PROJECT_ROOT / "data" / "processed" / "v2" / "growth_shrinkage_v24"
print("PROJECT_ROOT:", PROJECT_ROOT, "| serial exists:", SERIAL.exists())

In [ ]:
# ~30 s on CPU. Writes v24_growth_shrinkage_audit.json and v24_crossfit_forecasts.csv
r = subprocess.run([sys.executable, "notebooks/17_growth_shrinkage_audit.py", str(SERIAL), str(OUTDIR)],
                   capture_output=True, text=True)
print(r.stderr[-3000:])
r.check_returncode()
audit = json.load(open(OUTDIR / "v24_growth_shrinkage_audit.json"))

In [ ]:
import pandas as pd
print("Primary split (seed 17), mean |log-VAF error|:")
display(pd.Series(audit["primary_seed17_mae_log"]).sort_values())
print("By cohort:")
display(pd.DataFrame(audit["primary_seed17_by_cohort"]).T.round(4))
print("20 repeated CV seeds [mean, min, max]:")
display(pd.DataFrame(audit["repeated_cv_20seeds"], index=["mean","min","max"]).T.sort_values("mean"))
print("Leave-one-cohort-out:")
display(pd.DataFrame(audit["loco_by_cohort"]).T.round(4))

In [ ]:
b = pd.DataFrame(audit["bootstrap_delta_mean_lo_hi_pbetter"], index=["delta","ci_lo","ci_hi","p_better"]).T
keep = [k for k in b.index if k.endswith("minus_carry_forward") or k.endswith("minus_global_alpha")]
display(b.loc[keep].round(4))
print("VAF-threshold sensitivity [mae, delta_vs_cf, lo, hi, p]:")
display(pd.DataFrame(audit["vaf_threshold_sensitivity_mae_log_and_delta_vs_cf"]).T)
print("80% PI coverage:", audit["pi80_coverage_overall"])
display(pd.DataFrame(audit["pi80_coverage_by_cohort"]))

## Decision rule

A model is promoted only if it beats **carry-forward** and the v2.3 **global_alpha** shrinkage under patient-held-out validation, with a patient-cluster bootstrap CI excluding zero, stable across CV seeds, and not collapsing under leave-one-cohort-out. Negative results are reported, not hidden.